In [3]:
import sys
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project")
sys.path.append(r"C:\Users\Rakesh\Documents\summer-project\src")

from functions_v2 import *
from two_level_mc import TwoLevelSetup
from graph_mlmc_model import GraphTwoLevelModel
from mlmc_runner import MLMCRunner
from sksparse.cholmod import cholesky as sparse_cholesky

In [6]:
edges, n_vertices, edge_weights = load_graph(r"../../data/raw/oregon1_010526.txt")
num_components = check_connectivity(edges, n_vertices)
if num_components > 1:
    edges, n_vertices, edge_weights = filter_to_largest_component(edges, n_vertices, edge_weights)

A, D, L = build_graph_matrices(edges, n_vertices)
lambda_min = compute_lambda_min(L, D)
L_sigma = build_shifted_laplacian(L, D, lambda_min)

import networkx as nx
G_nx = nx.Graph()
G_nx.add_edges_from(edges)
gamma_in, gamma_out, diameter = find_diameter_endpoints(G_nx, n_sample=5, k_hop=4)

✓ Loaded: ../../data/raw/oregon1_010526.txt
  Vertices : 11174
  Edges    : 23409
  Weighted : no

Components: 1
  -> Graph is fully connected, safe to proceed

✓ Phase 1 complete: A, D, L built as sparse matrices
  Matrix size : 11174 x 11174
  Degree range: [1, 2389]
  Non-zeros in L: 57992

✓ lambda_min = 0.032899
  (eigenvalues found: [0.         0.03289937])
✓ Phase 2 complete: L_sigma built (sigma^2 = 0.25)
  L_sigma type: sparse



In [8]:
setup = TwoLevelSetup.build(
    edges, n_vertices, L_sigma, lambda_min,
    gamma_in, gamma_out,
    build_incidence_matrix, sparse_cholesky,
    max_size=10
)

gamma_in: 11, gamma_out: 92, n_vertices: 11174
Boundary fraction: 0.9218%
  -> Likely safe for aggregation (comparable to validated successes).
n_coarse: 6115 (6115 aggregates)
Size distribution -- min: 1, max: 10, mean: 1.83
Singletons: 4483 (73.3%)
gamma_in_coarse: 6, gamma_out_coarse: 74
Overlap (must be empty): set()
Interior coarse vertices: 6035 (98.7%)
coarse edges: 16624 (from 23409 fine edges)


In [23]:
model = GraphTwoLevelModel(setup)
runner = MLMCRunner(model, base_seed=0)
result = runner.run_fixed(samples_per_level=[200, 100])

print(f"Estimate: {result.estimate:.6f}")
print(f"Standard error: {result.standard_error:.6f}")

Estimate: 17.102688
Standard error: 0.028417


In [25]:
for lr in result.level_results:
    print(f"level={lr.level}, n={lr.sample_count}, mean_correction={lr.mean_correction:.6f}, "
          f"mean_cost={lr.mean_sample_cost:.6f}")

level=0, n=200, mean_correction=19.863826, mean_cost=0.098927
level=1, n=100, mean_correction=-2.761139, mean_cost=0.281187


In [33]:
from two_level_mc import *

In [35]:
import time

# Time YOUR direct approach
t0 = time.time()
qf, qc = run_one_paired_sample(setup, seed=0)
print(f"Direct: {time.time()-t0:.4f}s")

# Time THEIRS, single sample
from mlmc_correction import compute_sample_correction
rng = np.random.default_rng(0)
t0 = time.time()
correction = compute_sample_correction(model, fine_level=1, rng=rng)
print(f"Via framework: {time.time()-t0:.4f}s")
print(f"Reported elapsed_time: {correction.elapsed_time:.4f}s")

Direct: 0.3730s
Via framework: 0.3342s
Reported elapsed_time: 0.3328s


In [39]:
import time

runner2 = MLMCRunner(model, base_seed=1)  # fresh runner, different seed to avoid reusing prior samples

t0 = time.time()
result = runner2.run_fixed(samples_per_level=[200, 100])
mlmc_time = time.time() - t0
print(f"Full MLMCRunner run: {mlmc_time:.2f}s")
print(f"Estimate: {result.estimate:.6f}")

t0 = time.time()
result_direct = run_paired_validation(setup, N=100)
estimate_direct = two_level_estimate(setup, result_direct, N_coarse_only=200)
direct_time = time.time() - t0
print(f"\nFull direct run: {direct_time:.2f}s")
print(f"Estimate: {estimate_direct['estimate']:.6f}")

print(f"\nSlowdown factor: {mlmc_time/direct_time:.1f}x")

Full MLMCRunner run: 57.54s
Estimate: 17.045122


Paired samples: 100%|██████████| 100/100 [00:40<00:00,  2.45sample/s, Q_fine=17.0577, Q_coarse=19.8252]



N = 100 paired samples
Q_fine   : mean=17.057662  var=0.099102
Q_coarse : mean=19.825242  var=0.144558
Q_fine - Q_coarse : mean=-2.767580  var=0.005292
Correlation(Q_fine, Q_coarse): 0.9958
Variance reduction: 18.73x


Coarse-only samples: 100%|██████████| 200/200 [00:20<00:00,  9.63sample/s, Q_coarse=19.7810]


Coarse-only base estimate (N=200): 19.780969
Correction term mean (paired samples): -2.767580
Two-level estimate of E[Q_fine]: 17.013389
Direct fine-only mean (for comparison): 17.057662

Full direct run: 61.69s
Estimate: 17.013389

Slowdown factor: 0.9x
